<a href="https://colab.research.google.com/github/mouha-ndour/NLP-with-Transformers/blob/main/Text_Generation_with_Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**GREADY SEARCH DECODING**

The simplest decoding method to let discrete tokens from a model's continuous output is to greedily select the token with thee highest probability at each timestep.

In [1]:
!pip install torch

In [2]:
# To see how greedy search works, let's start by loading the 1.5-billion-parameter version
# of GPT-2 with a language modeling head.

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device="cuda" if torch.cuda.is_available() else "cpu"
model_name="gpt2-xl"
tokenizer=AutoTokenizer.from_pretrained(model_name)
model=AutoModelForCausalLM.from_pretrained(model_name).to(device)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/6.43G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [8]:
import pandas as pd

input_txt="Transformers are the"
input_ids=tokenizer(input_txt, return_tensors="pt")["input_ids"].to(device)
iterations=[]
n_steps=8
choices_per_step=5

with torch.no_grad():
  for _ in range(n_steps):
    iteration=dict()
    iteration["Input"]=tokenizer.decode(input_ids[0])
    output=model(input_ids=input_ids)
    # Select logits of the first batch and the last token
    next_token_logits=output.logits[0,-1, :] # Corrected: Use -1 to get logits for the last token
    next_token_probs=torch.softmax(next_token_logits, dim=-1) # Corrected: dim should be -1 for a 1D tensor
    sorted_ids=torch.argsort(next_token_probs, dim=-1, descending=True) # Corrected: dim should be -1 and typo 'descending'

    # Store tokens with highest probabilities
    for choice_idx in range(choices_per_step):
      token_id=sorted_ids[choice_idx]
      token_prob=next_token_probs[token_id].cpu().numpy()
      token_choice=(
          f"{tokenizer.decode(token_id)} ({100*token_prob:.2f}%)"
      )
      iteration[f"Choice {choice_idx}"]= token_choice

    # Append the completed iteration data for the current step
    iterations.append(iteration) # Corrected placement: append once per outer loop

    # Append predicted next token (the greedy choice) to input for the next step
    input_ids=torch.cat([input_ids, sorted_ids[0].unsqueeze(0).unsqueeze(0)], dim=1) # Corrected: Reshape top token to (1,1) for concatenation

In [9]:
pd.DataFrame(iterations)

,Input,Choice 0,Choice 1,Choice 2,Choice 3,Choice 4
0,Transformers are the,most (8.53%),only (4.96%),best (4.65%),Transformers (4.37%),ultimate (2.16%)
1,Transformers are the most,popular (16.78%),powerful (5.37%),common (4.96%),famous (3.72%),successful (3.20%)
2,Transformers are the most popular,toy (10.63%),toys (7.23%),Transformers (6.60%),of (5.46%),and (3.76%)
3,Transformers are the most popular toy,line (34.38%),in (18.20%),of (11.71%),brand (6.10%),line (2.69%)
4,Transformers are the most popular toy line,in (46.28%),of (15.09%),", (4.94%)",on (4.40%),ever (2.72%)
5,Transformers are the most popular toy line in,the (65.99%),history (12.42%),America (6.91%),Japan (2.44%),North (1.40%)
6,Transformers are the most popular toy line in the,world (69.26%),United (4.55%),history (4.29%),US (4.23%),U (2.30%)
7,Transformers are the most popular toy line in ...,", (39.73%)",. (30.64%),and (9.87%),with (2.32%),today (1.74%)


In [ ]:
# Now let's generate some text. Although transformers provides generate()
# function for autoregressive models like GPT-2, we'll implement
# this decoding method.
n_steps=8 #
input_txt="Transformers are the"
input_ids=tokenizer(input_txt, return_tensors="pt")["input_ids"].to(device)
output=model.generate(input_ids, max_new_tokens=n_steps,do_sample=False)
print(tokenizer.decode(output[0]))


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Transformers are the most popular toy line in the world,
